# Capítulo 4 — Razonar con probabilidades · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch04_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin dar error. La solución está en `ch04_corrige.ipynb`, en la misma página complementaria.

**Índice.** Datos · Los cálculos de las lecciones 4.1 a 4.5 · Ejercicios 4.1 a 4.6

**Antes de cada cálculo**, pregúntate en qué conjunto estás contando: spams, mensajes marcados o mensajes recibidos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLUE, ORANGE = '#2a78d6', '#eb6834'   # «ningún spam se escapa», curva de la distribución normal
from math import comb

## Datos

Los identificadores D4.1 y D4.2 son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 4, copiados de los archivos CSV del libro (no modificar)
# Filas: spam, legítimo. Columnas: marcado, no marcado.
ROWS, COLUMNS = ['spam', 'legítimo'], ['marcado', 'no marcado']

# D4.1 — buzón donde el 20 % de los mensajes son spam (1 000 mensajes)
inbox_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=ROWS, columns=COLUMNS)

# D4.2 — mismo filtro, buzón donde el 2 % de los mensajes son spam (1 000 mensajes)
inbox_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=ROWS, columns=COLUMNS)

# D4.3 — 1 000 tiempos de procesamiento (ms), regenerados idénticos al archivo CSV
durations = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `inbox_20` | D4.1: buzón donde el 20 % de los mensajes son spam (filas spam / legítimo, columnas marcado / no marcado) | mensajes | — |
| `inbox_2` | D4.2: mismo filtro, buzón donde el 2 % de los mensajes son spam | mensajes | — |
| `durations` | D4.3: 1000 tiempos de procesamiento | ms | — |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 4.1 a 4.5.

In [ ]:
# Lección 4.1 — contar y luego dividir por el grupo correcto (ecuación 4.1)
t = inbox_20
n = t.values.sum()
print('total:', n, '| spams:', t.loc['spam'].sum(), '| marcados:', t['marcado'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(marcado) =', t['marcado'].sum() / n,
      '| P(spam y marcado) =', t.loc['spam', 'marcado'] / n)
# Entre los marcados: el denominador es la columna «marcado»
print(f"spams entre los marcados: {t.loc['spam', 'marcado']} / {t['marcado'].sum()} "
      f"≈ {t.loc['spam', 'marcado'] / t['marcado'].sum():.2f}")

In [ ]:
# Lección 4.2 — el mismo filtro en un buzón donde los spams son raros (ecuación 4.2)
d, f = 0.90, 0.05                   # proporciones marcadas: de los spams, de los mensajes legítimos
for p, t in [(0.20, inbox_20), (0.02, inbox_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    rebuilt = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(rebuilt) == t.values).all()   # la tabla se deduce de las tasas
    print(f'tasa base {p:.0%}: {t.values.tolist()}  →  proporción de spams entre los marcados '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lección 4.3 — dos probabilidades condicionadas, dos denominadores (ecuación 4.3)
def conditional(t, case, group):
    """Proporción de la casilla (fila, columna) dentro del grupo: una fila ('spam') o una columna ('marcado')."""
    denominator = t.loc[group].sum() if group in t.index else t[group].sum()
    return t.loc[case] / denominator, denominator

for name, t in [('20 %', inbox_20), ('2 %', inbox_2)]:
    recall, den_r = conditional(t, ('spam', 'marcado'), 'spam')
    precision, den_p = conditional(t, ('spam', 'marcado'), 'marcado')
    print(f'buzón con {name}: P(marcado | spam) = {recall:.2f} (denominador {den_r}); '
          f'P(spam | marcado) = {precision:.2f} (denominador {den_p})')

# Los mismos números con scikit-learn, a partir de etiquetas mensaje por mensaje (si está instalado)
try:
    from sklearn.metrics import precision_score, recall_score
    truth = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predicted_label = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = marcado
    print(f'scikit-learn: exhaustividad = {recall_score(truth, predicted_label):.2f}, precisión = {precision_score(truth, predicted_label):.2f}')
except ImportError:
    print('scikit-learn no está instalado: este cálculo es opcional.')

In [ ]:
# Lección 4.4 — el teorema de Bayes (ecuaciones 4.4 y 4.5) y la curva del libro
def bayes(p, d=0.90, f=0.05):
    p_flagged = d * p + f * (1 - p)                  # ecuación 4.5
    return d * p / p_flagged                        # ecuación 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p}: P(marcado) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | marcado) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=FLAGGED, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Tasa base: proporción de spams (%)'); plt.ylabel('P(spam | marcado)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lección 4.5 — diez spams: distribución binomial (ecuación 4.7) y suceso complementario (ecuación 4.8)
n, p = 10, 0.1                        # contamos sobre 10 mensajes SPAM; p = probabilidad de que se escape
distribution = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| al menos uno se escapa =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 a 4:', [round(v, 3) for v in distribution[:5]], '| suma =', round(sum(distribution), 6))
colors = [BLUE] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), distribution, color=colors, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Número de spams que se escapan entre 10 spams'); plt.ylabel('Probabilidad'); plt.show()

# Un vistazo a la distribución normal: tiempos de procesamiento (D4.3)
m, s = durations.mean(), durations.std()
print(f'media = {m:.2f} ms, desviación típica = {s:.2f} ms; a ±1 desviación típica: {np.mean(abs(durations - m) <= s):.1%}, '
      f'a ±2 desviaciones típicas: {np.mean(abs(durations - m) <= 2 * s):.1%}')

## Ejercicio 4.1 — Leer un gráfico: veinte spams

*Nociones: distribución binomial, suceso complementario (lección 4.5).* Primero lee la figura del libro.

### a) a d) Tus lecturas

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: la probabilidad binomial P(X = k) (ecuación 4.7)
def binomial(n, p, k):
    return None   # PARA COMPLETAR; comb(n, k) da el coeficiente binomial

In [ ]:
# Comprobación con los 20 spams del ejercicio
if binomial(20, 0.05, 0) is None:
    print('Completa la función.')
else:
    p0, p1 = binomial(20, 0.05, 0), binomial(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, al menos uno = {1 - p0:.4f} →',
          'correcto' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'revisar')

## Ejercicio 4.2 — Un filtro más prudente

*Nociones: probabilidad condicionada (lección 4.3), teorema de Bayes (lección 4.4), tasa base (lección 4.2).* El filtro prudente marca el 75 % de los spams y el 1 % de los mensajes legítimos. Buzón de 10 000 mensajes, de los cuales el 10 % son spams. Haz primero las preguntas a) a e) en papel.

### a) a c) En papel

*Tus respuestas:* …

### a) La tabla en Python

In [ ]:
# PARA COMPLETAR: los cuatro recuentos de la tabla, para 10 000 mensajes
spam_flagged = None
spam_not_flagged = None
legit_flagged = None
legit_not_flagged = None

In [ ]:
# Comprobación
cells = [spam_flagged, spam_not_flagged, legit_flagged, legit_not_flagged]
if None in cells:
    print('Completa la celda anterior.')
else:
    cautious = pd.DataFrame([cells[:2], cells[2:]], index=ROWS, columns=COLUMNS)
    print(cautious)
    print('tabla:', 'correcto' if cells == [750, 250, 90, 8910] else 'revisar')

### b) Las dos probabilidades condicionadas

In [ ]:
# PARA COMPLETAR: ¿en qué grupo cuentas? Escribe el cociente con los recuentos.
p_flagged_if_spam = None
p_spam_if_flagged = None

In [ ]:
# Comprobación
for name, yours, truth in [('P(marcado | spam)', p_flagged_if_spam, 0.75),
                            ('P(spam | marcado)', p_spam_if_flagged, 750 / 840)]:
    print(f'{name}:', 'aún sin completar' if yours is None else
          ('correcto' if abs(yours - truth) < 0.001 else 'revisar'))

### c) a e) El teorema de Bayes como función

In [ ]:
# PARA COMPLETAR: P(spam | marcado) a partir de la tasa base p y de las tasas del filtro d y f
def flagged_spam_probability(p, d, f):
    return None   # PARA COMPLETAR (ecuaciones 4.4 y 4.5)

In [ ]:
# Comprobación de c), d) y e)
if flagged_spam_probability(0.1, 0.75, 0.01) is None:
    print('Completa la función.')
else:
    scenario = [('c) prudente, 10 % de spams', (0.10, 0.75, 0.01), 0.8929),
           ('d) prudente, 1 % de spams', (0.01, 0.75, 0.01), 0.4310),
           ('e) prudente, 2 % de spams', (0.02, 0.75, 0.01), 0.6048),
           ('e) filtro de las lecciones, 2 % de spams', (0.02, 0.90, 0.05), 0.2687)]
    for name, args, expected in scenario:
        v = flagged_spam_probability(*args)
        print(f'{name}: {v:.4f} →', 'correcto' if abs(v - expected) < 0.0001 else 'revisar')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, flagged_spam_probability(P, 0.90, 0.05), color=FLAGGED, linewidth=2,
             label='filtro de las lecciones (90 %; 5 %)')
    plt.plot(100 * P, flagged_spam_probability(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='filtro prudente (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Tasa base: proporción de spams (%)'); plt.ylabel('P(spam | marcado)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) y e) Tus respuestas

*Tu predicción para d), antes de calcular:* …

*Tu elección en e):* …

## Ejercicio 4.3 — Tres spams y luego diez mensajes

*Nociones: ensayo de Bernoulli y distribución binomial (lección 4.5), probabilidad condicionada (lección 4.3).* Haz primero las preguntas en papel.

### a) a e) En papel

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: las probabilidades de 0, 1, 2, 3 spams que se escapan entre 3, y luego la respuesta d)
distribution_3 = None        # lista de cuatro números
p_d = None          # al menos un spam se escapa entre 10 mensajes recibidos

In [ ]:
# Comprobación
if distribution_3 is None or p_d is None:
    print('Completa la celda anterior.')
else:
    print('b):', [round(v, 3) for v in distribution_3], '→', 'correcto' if np.allclose(distribution_3, [0.729, 0.243, 0.027, 0.001]) else 'revisar')
    print(f'd): {p_d:.4f} →', 'correcto' if abs(p_d - 0.1829) < 0.0001 else 'revisar')

## Ejercicio 4.4 — Simular diez spams

*Nociones: distribución binomial (lección 4.5), frecuencia y probabilidad.* Contamos sobre **diez spams**.

In [ ]:
# PARA COMPLETAR: una serie de 10 spams, cada uno se escapa con probabilidad 0,1
def series_simulation(rng):
    """Devuelve el número de spams que se escapan entre 10. rng.random() saca un número entre 0 y 1."""
    return None   # PARA COMPLETAR

In [ ]:
# b) y c) 10 000 series (semilla fija para obtener los mismos números)
rng = np.random.default_rng(4)
if series_simulation(rng) is None:
    print('Completa la función.')
else:
    rng = np.random.default_rng(4)
    X = np.array([series_simulation(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'frecuencia observada: {freq:.4f}; probabilidad calculada: {1 - 0.9 ** 10:.4f} →',
          'correcto' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'revisar')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=FLAGGED, label='frecuencia observada')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probabilidad (ecuación 4.7)')
    plt.xlabel('Spams que se escapan entre 10'); plt.legend(); plt.show()
    # d) Más series: la frecuencia se acerca a la probabilidad
    for N in (100, 100, 100, 100000):
        Y = np.array([series_simulation(rng) for _ in range(N)])
        print(f'{N:>6} series: frecuencia = {np.mean(Y >= 1):.4f}')

### d) Tu interpretación

*Tu respuesta:* …

## Ejercicio 4.5 — Predecir y luego comprobar: una campaña de spam

*Nociones: independencia, suceso complementario (lección 4.5).*

### a) a c) Tus predicciones y cálculos

*Tu predicción:* …

In [ ]:
# PARA COMPLETAR: una campaña de 10 spams idénticos, que se escapan todos juntos con probabilidad 0,1
def campaign_simulation(rng):
    return None   # PARA COMPLETAR: devuelve 10 o 0

In [ ]:
# d) Comprobación por simulación
rng = np.random.default_rng(5)
if campaign_simulation(rng) is None:
    print('Completa la función.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([campaign_simulation(rng) for _ in range(10000)])
    print(f'al menos uno se escapa: {np.mean(Z >= 1):.4f}; media: {Z.mean():.3f} →',
          'correcto' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'revisar')

## Ejercicio 4.6 — Desafío: ¿cuántos spams hacen falta para superar un 99 % de riesgo de que se escape uno?

*Nociones: distribuciones de Bernoulli y binomial, distribución normal (lección 4.5), desviación típica (lección 1.1).*

### a) a d) Tus demostraciones y cálculos

*Tus respuestas:* …

In [ ]:
# PARA COMPLETAR: b) el menor n tal que 0,9 ** n < 0,01; c) media y desviación típica para n = 100
n_min = None
mean_100 = None
std_100 = None

In [ ]:
# Comprobación y luego d) binomial frente a normal
if None in (n_min, mean_100, std_100):
    print('Completa la celda anterior.')
else:
    print(f'n_min = {n_min} →', 'correcto' if n_min == 44 else 'revisar')
    print(f'media = {mean_100:.1f}, desviación típica = {std_100:.1f} →',
          'correcto' if np.isclose(mean_100, 10) and np.isclose(std_100, 3) else 'revisar')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='distribución binomial (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='distribución normal (media 10, desviación típica 3)')
    plt.xlabel('Spams que se escapan entre 100'); plt.legend(); plt.show()